# Video Translation Pipeline - Debug Notebook

This notebook provides an interactive debugging environment for the cloud video translation pipeline.
Each stage of the pipeline is broken into separate cells, allowing you to:

- Inspect intermediate outputs at each stage
- Debug translation quality issues
- Test different configurations
- Understand how each component works

## Pipeline Overview

1. **Setup & Configuration** - Load dependencies and API clients
2. **Input Configuration** - Set video URL and target language
3. **Video Info Retrieval** - Fetch metadata without downloading
4. **Video Download** - Download and optionally trim the video
5. **Audio Extraction** - Extract audio for processing
6. **Audio Separation** - Separate vocals from background music
7. **Speaker Diarization** - Identify different speakers
8. **Voice Sample Extraction** - Extract samples for voice cloning
9. **Transcription** - Convert speech to text
10. **Translation** - Translate text to target language
11. **Speech Synthesis** - Generate translated speech
12. **Audio Mixing** - Mix speech with background
13. **Subtitle Generation** - Create SRT subtitles
14. **Final Merge** - Combine everything into final video

## Cell 1: Setup and Configuration

This cell loads all required dependencies and initializes API clients.

### Dependencies

- **yt-dlp**: Downloads videos from YouTube and 1700+ sites
- **openai**: OpenAI Whisper API for speech-to-text transcription
- **replicate**: Replicate API for:
  - Demucs (audio separation)
  - Speaker diarization
  - Llama (LLM translation)
  - Chatterbox (TTS with voice cloning)
- **deep_translator**: Google Translate fallback for translation
- **httpx**: Async HTTP client for file downloads
- **numpy**: Audio data manipulation

### Environment Variables

Required API keys (loaded from `.env`):
- `OPENAI_API_KEY`: For Whisper transcription
- `REPLICATE_API_TOKEN`: For all Replicate models

Optional configuration:
- `OUTPUT_DIR`: Where to save output files (default: `/tmp/yt-translate`)
- `OXYLABS_PROXY`: Proxy for YouTube downloads (to avoid rate limits)
- `YOUTUBE_COOKIES`: Netscape cookie file for authenticated downloads

In [ ]:
# =============================================================================
# Setup and Configuration
# =============================================================================

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

# Standard library imports
import asyncio
import os
import re
import subprocess
import tempfile
import time
from pathlib import Path
from typing import Optional

# Third-party imports
import httpx
import numpy as np
import replicate
from replicate.helpers import FileOutput
import yt_dlp
from deep_translator import GoogleTranslator
from openai import OpenAI

# Project imports - language mappings
from languages import (
    SUPPORTED_LANGUAGES,
    GOOGLE_LANG_CODES,
    ISO_639_1_TO_639_2,
    LANG_NAMES,
    get_language_name,
    get_google_code,
    get_iso_639_2_code,
)

# Project imports - configuration constants
from config import (
    GOOGLE_TRANSLATE_CHUNK_SIZE,
    LLM_MAX_SEGMENTS_PER_BATCH,
    LLM_BATCH_OVERLAP,
    LLM_MAX_TOKENS,
    LLM_TEMPERATURE,
    CHATTERBOX_SAMPLE_RATE,
    MIN_VOICE_SAMPLE_DURATION,
    MAX_VOICE_SAMPLE_DURATION,
    SIGNED_URL_EXPIRATION,
    REPLICATE_MODEL_LLAMA,
    REPLICATE_MODEL_CHATTERBOX,
    REPLICATE_MODEL_DEMUCS,
    REPLICATE_MODEL_DIARIZATION,
)

# Project imports - audio utilities
from utils.audio_helpers import read_wav_file, write_wav_file, resample_audio

# =============================================================================
# Helper Functions from cloud_translate.py
# =============================================================================

def parse_timestamp(ts: str) -> float:
    """
    Convert a timestamp string to float seconds.
    
    Handles format like "0:00:00.497812" (H:MM:SS.microseconds)
    or "1:30:45.5" (H:MM:SS.fraction).
    
    Args:
        ts: Timestamp string in format "H:MM:SS.fraction" or "H:MM:SS"
    
    Returns:
        Time in seconds as float (e.g., "1:30:45.5" -> 5445.5)
    """
    parts = ts.split(":")
    if len(parts) == 3:
        hours = int(parts[0])
        minutes = int(parts[1])
        seconds = float(parts[2])
        return hours * 3600 + minutes * 60 + seconds
    elif len(parts) == 2:
        minutes = int(parts[0])
        seconds = float(parts[1])
        return minutes * 60 + seconds
    else:
        return float(ts)


def format_timestamp_srt(seconds: float) -> str:
    """
    Convert seconds to SRT timestamp format (HH:MM:SS,mmm).
    
    Args:
        seconds: Time in seconds
    
    Returns:
        Timestamp string in SRT format
    """
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int((seconds % 1) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"


# =============================================================================
# Configuration from Environment
# =============================================================================

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
REPLICATE_API_TOKEN = os.getenv("REPLICATE_API_TOKEN")

# Output directory
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "/tmp/yt-translate"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Proxy configuration (optional)
OXYLABS_PROXY = os.getenv("OXYLABS_PROXY")
YOUTUBE_COOKIES = os.getenv("YOUTUBE_COOKIES")

# =============================================================================
# Initialize API Clients
# =============================================================================

# OpenAI client for Whisper API
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

# Replicate client is initialized via environment variable automatically
# It reads REPLICATE_API_TOKEN from the environment

# =============================================================================
# yt-dlp Configuration Helper
# =============================================================================

def get_ytdlp_opts(extra_opts: dict = None) -> dict:
    """Get yt-dlp options with optional proxy and cookie support."""
    opts = {'quiet': True, 'no_warnings': True}
    if OXYLABS_PROXY:
        opts['proxy'] = OXYLABS_PROXY
    if YOUTUBE_COOKIES:
        # Write cookies to temp file for yt-dlp
        cookie_file = OUTPUT_DIR / "youtube_cookies.txt"
        cookie_file.write_text(YOUTUBE_COOKIES)
        opts['cookiefile'] = str(cookie_file)
    if extra_opts:
        opts.update(extra_opts)
    return opts


# =============================================================================
# Display Configuration Status
# =============================================================================

print("=" * 60)
print("CONFIGURATION STATUS")
print("=" * 60)
print()
print(f"Output Directory: {OUTPUT_DIR}")
print(f"  - Exists: {OUTPUT_DIR.exists()}")
print()
print("API Keys:")
print(f"  - OPENAI_API_KEY: {'Set (' + OPENAI_API_KEY[:8] + '...)' if OPENAI_API_KEY else 'NOT SET'}")
print(f"  - REPLICATE_API_TOKEN: {'Set (' + REPLICATE_API_TOKEN[:8] + '...)' if REPLICATE_API_TOKEN else 'NOT SET'}")
print()
print("API Clients:")
print(f"  - OpenAI Client: {'Initialized' if openai_client else 'NOT INITIALIZED (missing API key)'}")
print(f"  - Replicate: {'Available' if REPLICATE_API_TOKEN else 'NOT AVAILABLE (missing API token)'}")
print()
print("Optional Configuration:")
print(f"  - Proxy (OXYLABS_PROXY): {'Configured' if OXYLABS_PROXY else 'Not configured'}")
print(f"  - YouTube Cookies: {'Configured' if YOUTUBE_COOKIES else 'Not configured'}")
print()
print("Replicate Models:")
print(f"  - LLM (Llama): {REPLICATE_MODEL_LLAMA}")
print(f"  - TTS (Chatterbox): {REPLICATE_MODEL_CHATTERBOX.split(':')[0]}")
print(f"  - Audio Separation (Demucs): {REPLICATE_MODEL_DEMUCS.split(':')[0]}")
print(f"  - Speaker Diarization: {REPLICATE_MODEL_DIARIZATION.split(':')[0]}")
print()
print("Pipeline Constants:")
print(f"  - Chatterbox Sample Rate: {CHATTERBOX_SAMPLE_RATE} Hz")
print(f"  - Min Voice Sample Duration: {MIN_VOICE_SAMPLE_DURATION} seconds")
print(f"  - Max Voice Sample Duration: {MAX_VOICE_SAMPLE_DURATION} seconds")
print(f"  - LLM Max Segments per Batch: {LLM_MAX_SEGMENTS_PER_BATCH}")
print(f"  - LLM Batch Overlap: {LLM_BATCH_OVERLAP}")
print(f"  - LLM Temperature: {LLM_TEMPERATURE}")
print()
print(f"Supported Languages: {len(SUPPORTED_LANGUAGES)}")
print(f"  {', '.join(SUPPORTED_LANGUAGES.values())}")
print()
print("=" * 60)
print("Setup complete! Proceed to the next cell to configure input.")
print("=" * 60)

## Cell 2: Input Configuration

This cell defines the input parameters for the pipeline. Modify these values to test different videos and configurations.

### Input Variables

| Variable | Description | Example |
|----------|-------------|--------|
| `VIDEO_URL` | URL of the video to translate | YouTube, Vimeo, Rumble, and 1700+ sites supported |
| `TARGET_LANGUAGE` | 2-letter ISO 639-1 language code | `"es"` (Spanish), `"ja"` (Japanese), `"fr"` (French) |
| `DURATION_LIMIT` | Max video length in seconds (None = full video) | `60` for 1 minute, `None` for no limit |
| `USE_MULTI_SPEAKER` | Enable per-speaker voice cloning | `True` for multiple speakers, `False` for single voice |

### Supported URL Formats

- **YouTube**: `https://www.youtube.com/watch?v=VIDEO_ID` or `https://youtu.be/VIDEO_ID`
- **Vimeo**: `https://vimeo.com/VIDEO_ID`
- **Rumble**: `https://rumble.com/VIDEO_ID.html`
- **Many more**: yt-dlp supports 1700+ sites ([full list](https://github.com/yt-dlp/yt-dlp/blob/master/supportedsites.md))

### Supported Language Codes

| Code | Language | Code | Language | Code | Language |
|------|----------|------|----------|------|----------|
| `ar` | Arabic | `he` | Hebrew | `no` | Norwegian |
| `zh` | Chinese | `hi` | Hindi | `pl` | Polish |
| `da` | Danish | `it` | Italian | `pt` | Portuguese |
| `nl` | Dutch | `ja` | Japanese | `ru` | Russian |
| `en` | English | `ko` | Korean | `es` | Spanish |
| `fi` | Finnish | `ms` | Malay | `sw` | Swahili |
| `fr` | French | `sv` | Swedish | `tr` | Turkish |
| `de` | German | `el` | Greek | | |

### Multi-Speaker Mode

- **`USE_MULTI_SPEAKER = True`**: Uses speaker diarization to identify different speakers, then clones each speaker's voice for synthesis. Best for interviews, podcasts, and multi-person content.
- **`USE_MULTI_SPEAKER = False`**: Uses a single voice for all speech synthesis. Faster and simpler, best for single-speaker content like tutorials or narration.

In [ ]:
# =============================================================================
# Input Configuration - Modify these values for your video
# =============================================================================

# Video URL to translate
# Supports: YouTube, Vimeo, Rumble, and 1700+ sites via yt-dlp
VIDEO_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"  # Example: Rick Astley

# Target language (2-letter ISO 639-1 code)
# Examples: "es" (Spanish), "ja" (Japanese), "fr" (French), "de" (German)
TARGET_LANGUAGE = "es"  # Spanish

# Duration limit in seconds (None = process full video)
# Useful for testing with long videos - set to 60 for first minute only
DURATION_LIMIT: int | None = 60  # Process only first 60 seconds

# Multi-speaker mode toggle
# True = Use speaker diarization and per-speaker voice cloning (better for interviews/podcasts)
# False = Single voice synthesis (faster, simpler, better for single-speaker content)
USE_MULTI_SPEAKER = True

# =============================================================================
# Display Configuration Summary
# =============================================================================

print("=" * 60)
print("INPUT CONFIGURATION")
print("=" * 60)
print()
print(f"Video URL: {VIDEO_URL}")
print(f"Target Language: {TARGET_LANGUAGE} ({get_language_name(TARGET_LANGUAGE)})")
print(f"Duration Limit: {DURATION_LIMIT} seconds" if DURATION_LIMIT else "Duration Limit: None (full video)")
print(f"Multi-Speaker Mode: {'Enabled' if USE_MULTI_SPEAKER else 'Disabled'}")
print()

# Validate target language
if TARGET_LANGUAGE not in SUPPORTED_LANGUAGES:
    print(f"WARNING: '{TARGET_LANGUAGE}' is not in supported languages!")
    print(f"   Supported: {list(SUPPORTED_LANGUAGES.keys())}")
else:
    print(f"Target language '{TARGET_LANGUAGE}' is supported")

print()
print("=" * 60)
print("Configuration complete! Proceed to video info retrieval.")
print("=" * 60)

## Cell 3: Video Info Retrieval

This cell fetches video metadata **without downloading** the actual video file.

### How it works

yt-dlp's `extract_info()` with `download=False` makes an API call to the video platform to retrieve metadata:

- **Title**: Video title as shown on the platform
- **Duration**: Length in seconds (needed for price calculation)
- **Video ID**: Platform-specific identifier (e.g., YouTube's 11-character ID)
- **Thumbnail URL**: Preview image URL

### Price Calculation

The estimated price is calculated using the formula:

```
price = ceil(duration_seconds / 60) * PRICE_PER_MINUTE
```

- Rounded up to the nearest minute
- Minimum charge is 1 minute
- Default rate: $0.50 per minute (configurable via `PRICE_PER_MINUTE_CENTS`)

### Error Handling

Common errors include:
- **Invalid URL**: Unsupported platform or malformed URL
- **Private/Restricted**: Video requires authentication or is geo-blocked
- **Removed/Unavailable**: Video no longer exists

In [ ]:
# =============================================================================
# Video Info Retrieval - Fetch metadata without downloading
# =============================================================================

# Price per minute in cents (configurable via environment variable)
PRICE_PER_MINUTE_CENTS = int(os.getenv("PRICE_PER_MINUTE_CENTS", "50"))  # $0.50 default


def calculate_price_cents(duration_seconds: int) -> int:
    """
    Calculate price in cents based on video duration.
    
    Price is calculated per minute, rounded up.
    Minimum price is 1 minute.
    """
    minutes = max(1, (duration_seconds + 59) // 60)  # Round up to nearest minute
    return minutes * PRICE_PER_MINUTE_CENTS


def format_duration(seconds: int) -> str:
    """Format duration in seconds to human-readable string (HH:MM:SS or MM:SS)."""
    hours = seconds // 3600
    minutes = (seconds % 3600) // 60
    secs = seconds % 60
    if hours > 0:
        return f"{hours}:{minutes:02d}:{secs:02d}"
    return f"{minutes}:{secs:02d}"


# =============================================================================
# Extract Video Info
# =============================================================================

video_info = None  # Will hold the extracted info for later cells

print("=" * 60)
print("VIDEO INFO RETRIEVAL")
print("=" * 60)
print()
print(f"Fetching info for: {VIDEO_URL}")
print()

try:
    with yt_dlp.YoutubeDL(get_ytdlp_opts()) as ydl:
        info = ydl.extract_info(VIDEO_URL, download=False)
        
        if info is None:
            raise ValueError("Could not extract video information from the provided URL")
        
        # Extract metadata
        title = info.get('title', 'Unknown Title')
        duration = info.get('duration')
        video_id = info.get('id', 'unknown')
        
        if duration is None:
            raise ValueError("Could not determine video duration. The URL may not point to a valid video.")
        
        # Get thumbnail (yt-dlp provides various thumbnail options)
        thumbnail = info.get('thumbnail')
        if not thumbnail:
            thumbnails = info.get('thumbnails', [])
            if thumbnails:
                thumbnail = thumbnails[-1].get('url')
        
        # Calculate price
        price_cents = calculate_price_cents(duration)
        
        # Store for later cells
        video_info = {
            'title': title,
            'duration': duration,
            'video_id': video_id,
            'thumbnail': thumbnail,
            'price_cents': price_cents,
            'uploader': info.get('uploader', 'Unknown'),
            'extractor': info.get('extractor', 'Unknown'),
        }
        
        # Display results
        print("Video Metadata")
        print("-" * 40)
        print(f"Title:      {title}")
        print(f"Video ID:   {video_id}")
        print(f"Duration:   {format_duration(duration)} ({duration} seconds)")
        print(f"Uploader:   {video_info['uploader']}")
        print(f"Platform:   {video_info['extractor']}")
        print()
        print(f"Thumbnail URL:")
        print(f"  {thumbnail}")
        print()
        print("Price Estimation")
        print("-" * 40)
        minutes = max(1, (duration + 59) // 60)
        print(f"Billable minutes: {minutes} (rounded up from {duration/60:.1f} min)")
        print(f"Rate: ${PRICE_PER_MINUTE_CENTS/100:.2f} per minute")
        print(f"Estimated price: ${price_cents/100:.2f} ({price_cents} cents)")
        print()
        
        # Note about duration limit
        if DURATION_LIMIT and duration > DURATION_LIMIT:
            limited_price_cents = calculate_price_cents(DURATION_LIMIT)
            print(f"With DURATION_LIMIT={DURATION_LIMIT}s: ${limited_price_cents/100:.2f}")
            print(f"  (Processing only first {format_duration(DURATION_LIMIT)} of {format_duration(duration)})")
        
        print()
        print("=" * 60)
        print("Video info retrieved successfully! Proceed to download.")
        print("=" * 60)

except yt_dlp.utils.DownloadError as e:
    print(f"ERROR: Invalid video URL or video not accessible")
    print(f"  Details: {str(e)}")
    print()
    print("Common causes:")
    print("  - URL is malformed or unsupported")
    print("  - Video is private or age-restricted")
    print("  - Video has been removed")
    print("  - Geographic restrictions apply")
    print()
    print("Try: Double-check the URL or use YouTube cookies for restricted videos.")

except Exception as e:
    print(f"ERROR: Failed to extract video information")
    print(f"  Exception type: {type(e).__name__}")
    print(f"  Details: {str(e)}")

## Cell 4: Video Download

This cell downloads the video file using yt-dlp and optionally trims it if `DURATION_LIMIT` is set.

### Download Process

1. **Format Selection**: yt-dlp selects `bestvideo+bestaudio/best` - the highest quality video and audio streams
2. **Merge Output**: Streams are merged into a single MP4 container using ffmpeg
3. **Duration Trimming**: If `DURATION_LIMIT` is set, ffmpeg trims the video using stream copy (fast, no re-encoding)

### Format Selection Details

The `bestvideo+bestaudio/best` format string means:
- Try to get the best video and best audio as separate streams, then merge them
- If that fails (e.g., audio-only or video-only), fall back to the single best stream

### Duration Trimming

When `DURATION_LIMIT` is set:
```bash
ffmpeg -i input.mp4 -t {DURATION_LIMIT} -c:v copy -c:a copy -y output.mp4
```

- `-t {DURATION_LIMIT}`: Stop writing output after this many seconds
- `-c:v copy -c:a copy`: Stream copy (no re-encoding) - very fast
- The original file is replaced with the trimmed version

### Output

- **video_path**: Path to the downloaded (and optionally trimmed) video file
- **File size**: Displayed in human-readable format (MB/GB)

In [ ]:
# =============================================================================
# Video Download - Download and optionally trim the video
# =============================================================================

def format_file_size(size_bytes: int) -> str:
    """Format file size in human-readable format."""
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024 * 1024:
        return f"{size_bytes / 1024:.1f} KB"
    elif size_bytes < 1024 * 1024 * 1024:
        return f"{size_bytes / (1024 * 1024):.1f} MB"
    else:
        return f"{size_bytes / (1024 * 1024 * 1024):.2f} GB"


# =============================================================================
# Download Video
# =============================================================================

video_path = None  # Will hold the path to the downloaded video

print("=" * 60)
print("VIDEO DOWNLOAD")
print("=" * 60)
print()

# Check prerequisite from previous cell
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
else:
    try:
        video_id = video_info['video_id']
        title = video_info['title']
        duration = video_info['duration']
        
        # Create job directory for this video
        job_dir = OUTPUT_DIR / video_id
        job_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"Downloading: {title}")
        print(f"Video ID: {video_id}")
        print(f"Output directory: {job_dir}")
        print()
        
        # Configure yt-dlp for download
        ydl_opts = get_ytdlp_opts({
            'format': 'bestvideo+bestaudio/best',
            'outtmpl': str(job_dir / f"{video_id}.%(ext)s"),
            'merge_output_format': 'mp4',
            'quiet': False,  # Show progress for debugging
            'no_warnings': False,
        })
        
        # Download the video
        print("Downloading video...")
        print("-" * 40)
        download_start = time.time()
        
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([VIDEO_URL])
        
        download_time = time.time() - download_start
        print("-" * 40)
        print(f"Download completed in {download_time:.1f} seconds")
        print()
        
        # Find the downloaded video file
        video_path = None
        for ext in ['mp4', 'mkv', 'webm']:
            p = job_dir / f"{video_id}.{ext}"
            if p.exists():
                video_path = p
                break
        
        if video_path is None:
            raise FileNotFoundError(f"Could not find downloaded video in {job_dir}")
        
        original_size = video_path.stat().st_size
        print(f"Downloaded file: {video_path.name}")
        print(f"Original size: {format_file_size(original_size)}")
        print()
        
        # Apply duration limit if specified
        if DURATION_LIMIT is not None and duration > DURATION_LIMIT:
            print(f"Applying duration limit: {DURATION_LIMIT} seconds")
            print(f"  (Trimming from {format_duration(duration)} to {format_duration(DURATION_LIMIT)})")
            print()
            
            trimmed_path = job_dir / f"{video_id}_trimmed.mp4"
            
            # Trim using ffmpeg with stream copy (fast, no re-encoding)
            trim_cmd = [
                'ffmpeg', '-i', str(video_path),
                '-t', str(DURATION_LIMIT),
                '-c:v', 'copy', '-c:a', 'copy',
                '-y', str(trimmed_path)
            ]
            
            print("Running ffmpeg trim...")
            trim_start = time.time()
            result = subprocess.run(trim_cmd, capture_output=True, text=True)
            trim_time = time.time() - trim_start
            
            if result.returncode != 0:
                print(f"ERROR: ffmpeg trim failed")
                print(f"  stderr: {result.stderr}")
            else:
                # Replace original with trimmed version
                video_path.unlink()  # Delete original
                # Rename trimmed to standard name
                final_path = job_dir / f"{video_id}.mp4"
                trimmed_path.rename(final_path)
                video_path = final_path
                
                trimmed_size = video_path.stat().st_size
                print(f"Trim completed in {trim_time:.1f} seconds")
                print(f"Trimmed size: {format_file_size(trimmed_size)}")
                print(f"Size reduction: {(1 - trimmed_size/original_size)*100:.1f}%")
        
        print()
        print("Download Summary")
        print("-" * 40)
        print(f"Video path: {video_path}")
        print(f"Final size: {format_file_size(video_path.stat().st_size)}")
        
        # Update video_info with actual processed duration
        actual_duration = min(duration, DURATION_LIMIT) if DURATION_LIMIT else duration
        video_info['actual_duration'] = actual_duration
        video_info['video_path'] = video_path
        print(f"Actual duration: {format_duration(actual_duration)} ({actual_duration} seconds)")
        
        print()
        print("=" * 60)
        print("Video downloaded successfully! Proceed to audio extraction.")
        print("=" * 60)
        
    except subprocess.CalledProcessError as e:
        print(f"ERROR: ffmpeg command failed")
        print(f"  Command: {' '.join(e.cmd)}")
        print(f"  Return code: {e.returncode}")
        if e.stderr:
            print(f"  stderr: {e.stderr}")
    
    except yt_dlp.utils.DownloadError as e:
        print(f"ERROR: Video download failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Network connectivity issues")
        print("  - Video is age-restricted (try using cookies)")
        print("  - Rate limiting (try using a proxy)")
        print("  - DRM-protected content")
    
    except Exception as e:
        print(f"ERROR: Unexpected error during download")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")

## Cell 5: Audio Extraction

This cell extracts audio from the downloaded video file for processing by the speech-to-text pipeline.

### Why 16kHz Mono PCM?

OpenAI's Whisper API works best with audio that matches its training data format:

- **16kHz Sample Rate**: Whisper was trained on 16kHz audio. Higher rates waste bandwidth without improving accuracy.
- **Mono (1 channel)**: Speech recognition doesn't benefit from stereo. Combining channels reduces file size by 50%.
- **PCM 16-bit**: Uncompressed format with no quality loss. The `.wav` extension signals this to downstream tools.

### ffmpeg Command Breakdown

```bash
ffmpeg -i video.mp4 -vn -acodec pcm_s16le -ar 16000 -ac 1 -y audio.wav
```

| Flag | Description |
|------|-------------|
| `-i video.mp4` | Input video file |
| `-vn` | No video output (audio only) |
| `-acodec pcm_s16le` | PCM 16-bit signed little-endian codec |
| `-ar 16000` | Set audio sample rate to 16kHz |
| `-ac 1` | Convert to mono (1 audio channel) |
| `-y` | Overwrite output file if exists |

### Output

- **audio_path**: Path to the extracted `.wav` file
- **Audio duration**: Should match video duration (or `DURATION_LIMIT` if set)
- **File size**: Approximately `duration_seconds * 32000` bytes (16000 Hz × 2 bytes × mono)

In [ ]:
# =============================================================================
# Audio Extraction - Extract audio from video for transcription
# =============================================================================

# Audio extraction parameters
WHISPER_SAMPLE_RATE = 16000  # 16kHz for Whisper API


def get_audio_duration(audio_path: Path) -> float:
    """
    Get the duration of an audio file using ffprobe.
    
    Args:
        audio_path: Path to the audio file
    
    Returns:
        Duration in seconds
    """
    cmd = [
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        str(audio_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip():
        return float(result.stdout.strip())
    return 0.0


# =============================================================================
# Extract Audio from Video
# =============================================================================

audio_path = None  # Will hold the path to the extracted audio

print("=" * 60)
print("AUDIO EXTRACTION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('video_path') is None:
    print("ERROR: video_path not available. Run the Video Download cell first.")
else:
    try:
        video_path = video_info['video_path']
        video_id = video_info['video_id']
        job_dir = video_path.parent
        
        # Define output audio path
        audio_path = job_dir / "audio.wav"
        
        print(f"Source video: {video_path}")
        print(f"Output audio: {audio_path}")
        print(f"Target format: {WHISPER_SAMPLE_RATE} Hz, mono, PCM 16-bit")
        print()
        
        # Build ffmpeg command for audio extraction
        ffmpeg_cmd = [
            'ffmpeg',
            '-i', str(video_path),        # Input video
            '-vn',                         # No video output
            '-acodec', 'pcm_s16le',       # PCM 16-bit signed little-endian
            '-ar', str(WHISPER_SAMPLE_RATE),  # 16kHz sample rate
            '-ac', '1',                    # Mono (1 channel)
            '-y',                          # Overwrite output file
            str(audio_path)
        ]
        
        print("Running ffmpeg...")
        print(f"  Command: {' '.join(ffmpeg_cmd)}")
        print()
        
        extract_start = time.time()
        result = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
        extract_time = time.time() - extract_start
        
        if result.returncode != 0:
            print(f"ERROR: ffmpeg audio extraction failed")
            print(f"  Return code: {result.returncode}")
            print(f"  stderr: {result.stderr}")
            audio_path = None
        else:
            # Verify the output file exists
            if not audio_path.exists():
                raise FileNotFoundError(f"Audio extraction failed: {audio_path} not created")
            
            # Get file info
            audio_size = audio_path.stat().st_size
            audio_duration = get_audio_duration(audio_path)
            
            # Store in video_info for later cells
            video_info['audio_path'] = audio_path
            video_info['audio_duration'] = audio_duration
            
            # Display results
            print("Audio Extraction Summary")
            print("-" * 40)
            print(f"Audio path: {audio_path}")
            print(f"File size: {format_file_size(audio_size)}")
            print(f"Duration: {format_duration(int(audio_duration))} ({audio_duration:.2f} seconds)")
            print(f"Sample rate: {WHISPER_SAMPLE_RATE} Hz")
            print(f"Channels: 1 (mono)")
            print(f"Bit depth: 16-bit")
            print(f"Extraction time: {extract_time:.2f} seconds")
            print()
            
            # Calculate expected file size and compare
            expected_size = int(audio_duration * WHISPER_SAMPLE_RATE * 2)  # 2 bytes per sample
            expected_size += 44  # WAV header
            print(f"Expected size (approx): {format_file_size(expected_size)}")
            print(f"Actual size: {format_file_size(audio_size)}")
            
            print()
            print("=" * 60)
            print("Audio extracted successfully! Proceed to audio separation.")
            print("=" * 60)
    
    except subprocess.CalledProcessError as e:
        print(f"ERROR: ffmpeg command failed")
        print(f"  Return code: {e.returncode}")
        if e.stderr:
            print(f"  stderr: {e.stderr}")
    
    except FileNotFoundError as e:
        print(f"ERROR: {str(e)}")
    
    except Exception as e:
        print(f"ERROR: Unexpected error during audio extraction")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")

## Cell 6: Audio Separation

This cell separates vocals from background music using Replicate's **Demucs** API. This improves transcription quality by removing non-speech audio.

### Why Separate Audio?

Background music, sound effects, and ambient noise can significantly degrade speech-to-text accuracy. Demucs is a state-of-the-art audio source separation model that isolates:

- **Vocals**: Human speech and singing
- **Drums**: Percussion elements
- **Bass**: Low-frequency instruments
- **Other**: Everything else (background music, effects)

For our pipeline, we keep **vocals** (for transcription) and **other** (as background to mix back in later).

### Demucs API Details

The Replicate model `cjwbw/demucs` accepts:

| Parameter | Description |
|-----------|-------------|
| `audio` | Base64 data URI of the audio file |
| `output_format` | Output format (`wav` recommended for quality) |

Returns a dictionary with URLs for each stem: `vocals`, `drums`, `bass`, `other`.

### Output Files

- **vocals.wav**: Isolated speech for transcription
- **background.wav**: Background audio (from "other" stem) for mixing back later

### Timing

API call timing depends on audio length and server load. Typical processing:
- 60 seconds audio: ~20-30 seconds
- 5 minutes audio: ~1-2 minutes

In [ ]:
# =============================================================================
# Audio Separation - Separate vocals from background using Demucs
# =============================================================================

import base64

# =============================================================================
# Audio Separation
# =============================================================================

vocals_path = None  # Will hold the path to isolated vocals
background_path = None  # Will hold the path to background audio

print("=" * 60)
print("AUDIO SEPARATION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('audio_path') is None:
    print("ERROR: audio_path not available. Run the Audio Extraction cell first.")
else:
    try:
        audio_path = video_info['audio_path']
        job_dir = audio_path.parent
        
        print(f"Source audio: {audio_path}")
        print(f"Model: {REPLICATE_MODEL_DEMUCS.split(':')[0]}")
        print()
        
        # Convert audio to base64 data URI for Replicate
        print("Encoding audio as base64 data URI...")
        with open(audio_path, "rb") as f:
            audio_data = base64.b64encode(f.read()).decode("utf-8")
        
        # Determine mime type based on file extension
        suffix = audio_path.suffix.lower()
        mime_type = "audio/wav" if suffix == ".wav" else "audio/mpeg"
        data_uri = f"data:{mime_type};base64,{audio_data}"
        
        print(f"  - Audio size: {format_file_size(len(audio_data))} (base64)")
        print(f"  - MIME type: {mime_type}")
        print()
        
        # Call Replicate Demucs API
        print("Calling Replicate Demucs API...")
        print("-" * 40)
        
        api_start = time.time()
        output = replicate.run(
            REPLICATE_MODEL_DEMUCS,
            input={
                "audio": data_uri,
                "output_format": "wav",
            }
        )
        api_time = time.time() - api_start
        
        print("-" * 40)
        print(f"API call completed in {api_time:.1f} seconds")
        print()
        
        # Define output paths
        vocals_path = job_dir / "vocals.wav"
        background_path = job_dir / "background.wav"
        
        # Handle FileOutput objects from Replicate v1.0+
        vocals_url = output["vocals"].url if isinstance(output["vocals"], FileOutput) else output["vocals"]
        other_url = output["other"].url if isinstance(output["other"], FileOutput) else output["other"]
        
        print("Downloading separated stems...")
        
        # Download vocals
        print(f"  - Downloading vocals...")
        download_start = time.time()
        
        # Use synchronous httpx for Jupyter compatibility
        with httpx.Client(timeout=None) as client:
            vocals_response = client.get(vocals_url)
            vocals_path.write_bytes(vocals_response.content)
            
            print(f"  - Downloading background (other stem)...")
            other_response = client.get(other_url)
            background_path.write_bytes(other_response.content)
        
        download_time = time.time() - download_start
        print(f"  - Download completed in {download_time:.1f} seconds")
        print()
        
        # Get file info
        vocals_size = vocals_path.stat().st_size
        background_size = background_path.stat().st_size
        
        # Store in video_info for later cells
        video_info['vocals_path'] = vocals_path
        video_info['background_path'] = background_path
        
        # Display results
        print("Audio Separation Summary")
        print("-" * 40)
        print(f"Vocals path: {vocals_path}")
        print(f"  - Size: {format_file_size(vocals_size)}")
        print()
        print(f"Background path: {background_path}")
        print(f"  - Size: {format_file_size(background_size)}")
        print()
        print("Timing")
        print("-" * 40)
        print(f"API call: {api_time:.1f} seconds")
        print(f"Download: {download_time:.1f} seconds")
        print(f"Total: {api_time + download_time:.1f} seconds")
        
        print()
        print("=" * 60)
        print("Audio separated successfully! Proceed to speaker diarization.")
        print("=" * 60)
    
    except replicate.exceptions.ReplicateError as e:
        print(f"ERROR: Replicate API call failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Invalid REPLICATE_API_TOKEN")
        print("  - Rate limiting")
        print("  - Model unavailable")
    
    except httpx.HTTPError as e:
        print(f"ERROR: Failed to download separated audio")
        print(f"  Details: {str(e)}")
    
    except Exception as e:
        print(f"ERROR: Unexpected error during audio separation")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")
        print()
        print("Fallback: Using original audio as vocals (no separation)")
        vocals_path = audio_path
        background_path = audio_path
        video_info['vocals_path'] = vocals_path
        video_info['background_path'] = background_path

## Cell 7: Speaker Diarization

This cell identifies different speakers in the audio using Replicate's **speaker-diarization** API. This is essential for multi-speaker content like interviews, podcasts, or panel discussions.

### What is Speaker Diarization?

Speaker diarization answers the question "who spoke when?" by segmenting audio into time ranges and assigning speaker labels. The model:

1. **Detects speech regions** - Identifies when someone is speaking vs. silence
2. **Clusters speakers** - Groups similar voice characteristics together
3. **Assigns labels** - Tags each segment with a speaker ID (SPEAKER_00, SPEAKER_01, etc.)

### Replicate API Details

The model `meronym/speaker-diarization` requires:

| Parameter | Description |
|-----------|-------------|
| `audio` | URL to the audio file (uploaded via `replicate.files.create()`) |

Returns a JSON object with segments:
```json
{
  "segments": [
    {"speaker": "A", "start": "0:00:00.5", "end": "0:00:05.2"},
    {"speaker": "B", "start": "0:00:05.5", "end": "0:00:10.1"},
    ...
  ]
}
```

### Speaker Label Format

The API returns single-letter labels (A, B, C...) which we convert to the standard format:
- `A` → `SPEAKER_00`
- `B` → `SPEAKER_01`
- etc.

### Output

- **diarization_segments**: List of segments with speaker labels and timestamps
- **Number of speakers detected**: Total unique speakers found
- **Segment table**: Formatted display of all segments

### Timing

Processing time depends on audio length:
- 60 seconds: ~10-20 seconds
- 5 minutes: ~30-60 seconds

In [ ]:
# =============================================================================
# Speaker Diarization - Identify different speakers in the audio
# =============================================================================

import json as json_module  # Avoid conflict with potential variable names

# =============================================================================
# Speaker Diarization
# =============================================================================

diarization_segments = []  # Will hold the parsed diarization segments

print("=" * 60)
print("SPEAKER DIARIZATION")
print("=" * 60)
print()

# Check prerequisites from previous cells
if video_info is None:
    print("ERROR: video_info not available. Run the Video Info Retrieval cell first.")
elif video_info.get('vocals_path') is None:
    print("ERROR: vocals_path not available. Run the Audio Separation cell first.")
else:
    try:
        vocals_path = video_info['vocals_path']
        job_dir = vocals_path.parent
        
        print(f"Source audio: {vocals_path}")
        print(f"Model: {REPLICATE_MODEL_DIARIZATION.split(':')[0]}")
        print()
        
        # Upload vocals file to Replicate for API access
        # The diarization model requires a URL, not base64 data URI
        print("Uploading vocals to Replicate...")
        upload_start = time.time()
        
        uploaded_file = replicate.files.create(file=vocals_path)
        vocals_url = uploaded_file.urls['get']
        
        upload_time = time.time() - upload_start
        print(f"  - Upload completed in {upload_time:.1f} seconds")
        print(f"  - File URL: {vocals_url[:80]}...")
        print()
        
        # Call Replicate speaker-diarization API
        print("Calling Replicate speaker-diarization API...")
        print("-" * 40)
        
        api_start = time.time()
        output = replicate.run(
            REPLICATE_MODEL_DIARIZATION,
            input={"audio": vocals_url}
        )
        api_time = time.time() - api_start
        
        print("-" * 40)
        print(f"API call completed in {api_time:.1f} seconds")
        print()
        
        # Handle different output types from Replicate v1.0+
        if isinstance(output, FileOutput):
            output_bytes = output.read()
            output = json_module.loads(output_bytes)
        elif isinstance(output, str):
            # Output is a URL pointing to JSON - fetch and parse
            with httpx.Client(timeout=None) as client:
                response = client.get(output)
                output = json_module.loads(response.text)
        
        # Parse output segments and convert speaker labels
        # Model returns: {"segments": [...], "speakers": {...}} or just a list
        segments_list = output.get("segments", []) if isinstance(output, dict) else output
        
        if not segments_list:
            print("WARNING: No speakers detected in audio")
            print("  The audio may not contain speech or the model couldn't detect it.")
            diarization_segments = []
        else:
            # Convert speaker labels from "A", "B" to "SPEAKER_00", "SPEAKER_01" format
            speaker_label_map = {}
            speaker_counter = 0
            
            for seg in segments_list:
                raw_label = seg.get("speaker", "")
                if raw_label not in speaker_label_map:
                    speaker_label_map[raw_label] = f"SPEAKER_{speaker_counter:02d}"
                    speaker_counter += 1
                
                # Parse timestamps - handle both string and numeric formats
                start_val = seg.get("start", "0")
                end_val = seg.get("stop", seg.get("end", "0"))
                
                start_sec = parse_timestamp(str(start_val)) if isinstance(start_val, str) else float(start_val)
                end_sec = parse_timestamp(str(end_val)) if isinstance(end_val, str) else float(end_val)
                
                diarization_segments.append({
                    "speaker": speaker_label_map[raw_label],
                    "start": start_sec,
                    "end": end_sec
                })
            
            # Store in video_info for later cells
            video_info['diarization_segments'] = diarization_segments
            video_info['num_speakers'] = speaker_counter
            
            # Display summary
            print("Diarization Summary")
            print("-" * 40)
            print(f"Number of speakers detected: {speaker_counter}")
            print(f"Total segments: {len(diarization_segments)}")
            print()
            
            # Calculate per-speaker statistics
            speaker_stats = {}
            for seg in diarization_segments:
                spk = seg['speaker']
                duration = seg['end'] - seg['start']
                if spk not in speaker_stats:
                    speaker_stats[spk] = {'count': 0, 'total_duration': 0.0}
                speaker_stats[spk]['count'] += 1
                speaker_stats[spk]['total_duration'] += duration
            
            print("Per-Speaker Statistics")
            print("-" * 40)
            for spk, stats in sorted(speaker_stats.items()):
                print(f"  {spk}: {stats['count']} segments, {stats['total_duration']:.1f}s total")
            print()
            
            # Display segment table
            print("Diarization Segments")
            print("-" * 60)
            print(f"{'#':<4} {'Speaker':<12} {'Start':>10} {'End':>10} {'Duration':>10}")
            print("-" * 60)
            
            for i, seg in enumerate(diarization_segments):
                duration = seg['end'] - seg['start']
                start_str = f"{seg['start']:.2f}s"
                end_str = f"{seg['end']:.2f}s"
                dur_str = f"{duration:.2f}s"
                print(f"{i:<4} {seg['speaker']:<12} {start_str:>10} {end_str:>10} {dur_str:>10}")
            
            print("-" * 60)
        
        print()
        print("Timing")
        print("-" * 40)
        print(f"File upload: {upload_time:.1f} seconds")
        print(f"API call: {api_time:.1f} seconds")
        print(f"Total: {upload_time + api_time:.1f} seconds")
        
        print()
        print("=" * 60)
        print("Speaker diarization complete! Proceed to voice sample extraction.")
        print("=" * 60)
    
    except replicate.exceptions.ReplicateError as e:
        print(f"ERROR: Replicate API call failed")
        print(f"  Details: {str(e)}")
        print()
        print("Common causes:")
        print("  - Invalid REPLICATE_API_TOKEN")
        print("  - Rate limiting")
        print("  - Model unavailable")
        print()
        print("Fallback: Continuing without diarization (single-speaker mode)")
        diarization_segments = []
        video_info['diarization_segments'] = []
        video_info['num_speakers'] = 0
    
    except httpx.HTTPError as e:
        print(f"ERROR: Failed to fetch diarization output")
        print(f"  Details: {str(e)}")
        diarization_segments = []
        video_info['diarization_segments'] = []
        video_info['num_speakers'] = 0
    
    except Exception as e:
        print(f"ERROR: Unexpected error during speaker diarization")
        print(f"  Exception type: {type(e).__name__}")
        print(f"  Details: {str(e)}")
        print()
        print("Fallback: Continuing without diarization (single-speaker mode)")
        diarization_segments = []
        video_info['diarization_segments'] = []
        video_info['num_speakers'] = 0